In [1]:
import numpy as np
import  pandas as pd
import librosa, librosa.display
import matplotlib.pyplot as plt
import os
import torch, torchaudio
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import IPython.display as ipd
import random
import seaborn as sns
from sklearn.model_selection import StratifiedShuffleSplit

# Data Loading Pipeline

In [2]:
BASE = "/kaggle/input/competitions/birdclef-2025"  # <-- CHANGE THIS to what FOUND printed
# example: BASE = "/kaggle/input/birdclef-2025/birdclef-2025" if needed

train = pd.read_csv(f"{BASE}/train.csv")
taxonomy = pd.read_csv(f"{BASE}/taxonomy.csv")
print(BASE, train.shape)

/kaggle/input/competitions/birdclef-2025 (28564, 13)


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
sample_df=train.sample(frac=0.1,random_state=42)

In [5]:
sample_df.shape

(2856, 13)

In [6]:
sample_df.head()

,primary_label,secondary_labels,type,filename,collection,rating,url,latitude,longitude,scientific_name,common_name,author,license
27685,yeofly1,[''],['call'],yeofly1/XC250351.ogg,XC,4.0,https://xeno-canto.org/250351,-4.4010,-79.8540,Tolmomyias sulphurescens,Yellow-olive Flycatcher,Niels Krabbe,cc-by-sa 4.0
25019,whbman1,[''],['mechanical sounds'],whbman1/XC268585.ogg,XC,5.0,https://xeno-canto.org/268585,-19.7323,-41.8141,Manacus manacus,White-bearded Manakin,Fabrice Schmitt,cc-by-nc-sa 4.0
96,21211,[''],[''],21211/XC913998.ogg,XC,3.0,https://xeno-canto.org/913998,4.0946,-52.6862,Allobates femoralis,Spotted-thighed Poison Frog,Delaitre guillaume,cc-by-nc-sa 4.0
5771,bugtan,[''],['flight call'],bugtan/XC480423.ogg,XC,5.0,https://xeno-canto.org/480423,-0.4383,-76.2791,Thraupis episcopus,Blue-gray Tanager,Jerome Fischer,cc-by-nc-sa 4.0
21879,strher,"['greani1', 'grekis']",['flight call'],strher/XC287867.ogg,XC,3.5,https://xeno-canto.org/287867,-22.4508,-42.7735,Butorides striata,Striated Heron,Jerome Fischer,cc-by-nc-sa 4.0


In [7]:
# Label map
labels = sorted(train['primary_label'].unique())
label2idx = {l:i for i,l in enumerate(labels)}
idx2label = {i:l for l,i in label2idx.items()}
print(f"Num classes: {len(labels)}")

class CFG:
    SR = 32000       # Sample rate (32kHz)
    DURATION = 5     # Every clip is fixed at 5 seconds
    N_MELS = 128     # 128 Mel frequency bins
    BATCH = 32       # Batch size

class BirdDataset(Dataset):
    def __init__(self, df):
        self.df = df
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=CFG.SR, n_mels=CFG.N_MELS, f_min=50, f_max=14000, n_fft=1024, hop_length=512
        )
        self.db = torchaudio.transforms.AmplitudeToDB()
        
    def __len__(self): return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(BASE, "train_audio", row['filename'])
        
        wav, sr = torchaudio.load(path)
        wav = wav.mean(dim=0) # mono
        
        # crop/pad to 5 sec
        if sr != CFG.SR:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=CFG.SR)
            wav = resampler(wav)
            
        target = CFG.SR * CFG.DURATION
        if len(wav) > target:
            start = torch.randint(0, len(wav)-target, (1,)).item()
            wav = wav[start:start+target]
        else:
            wav = F.pad(wav, (0, target - len(wav)))
            
        mel = self.mel(wav)
        mel = self.db(mel)
        mel = (mel - mel.mean()) / (mel.std() + 1e-6)
        
        label = label2idx[row['primary_label']]
        return mel, label


Num classes: 206


In [8]:
splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

In [9]:
counts = sample_df['primary_label'].value_counts()
rare_classes = counts[counts == 1].index

In [10]:
rare_classes.shape

(21,)

In [11]:
for i in rare_classes:
    print(i)

555086
spbwoo1
1194042
rosspo1
66531
1564122
65962
woosto
21116
963335
81930
65344
piwtyr1
566513
grysee1
555142
46010
plctan1
65419
22333
1462737


In [12]:
sample_df = sample_df[~sample_df['primary_label'].isin(rare_classes)].reset_index(drop=True)

In [13]:
sample_df.shape

(2835, 13)

In [14]:
train_idx, val_idx = next(splitter.split(sample_df, sample_df['primary_label']))

In [15]:
train_idx

array([ 539, 2806, 1330, ..., 2356, 2226,  837])

In [16]:
train_df = sample_df.iloc[train_idx].reset_index(drop=True)
val_df = sample_df.iloc[val_idx].reset_index(drop=True)

print(f"Train size: {len(train_df)} | Val size: {len(val_df)}")

Train size: 2268 | Val size: 567


In [17]:
train_dataset=BirdDataset(train_df)
val_dataset=BirdDataset(val_df)

In [18]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

In [19]:
mels, labs = next(iter(train_loader))
print("Batch shape:", mels.shape)       
print("Labels shape:", labs.shape)

Batch shape: torch.Size([32, 128, 313])
Labels shape: torch.Size([32])


#  The Timm Model

In [20]:
len(label2idx)

206

In [21]:
import timm
import torch

# 1. Define the number of classes (from your label dictionary)
num_classes = len(label2idx)

# 2. Create the model
# in_chans=1 adapts the first conv layer to accept 1 channel (our mel spectrogram)
# num_classes=206 changes the final fully connected layer to output 206 logits
model = timm.create_model(
    'tf_efficientnet_b0', 
    pretrained=True, 
    in_chans=1, 
    num_classes=num_classes
)

# 3. Move to GPU
model = model.to(device)

# 4. Print number of parameters (good habit to check)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded with {total_params:,} parameters")

# 5. Sanity check with a dummy tensor
# Create a fake batch of 2 spectrograms (Batch=2, Channels=1, Mel=128, Time=313)
dummy_input = torch.randn(2, 1, 128, 313).to(device)

# Pass it through the model
with torch.no_grad():
    dummy_output = model(dummy_input)

print(f"Dummy input shape: {dummy_input.shape}")
print(f"Dummy output shape: {dummy_output.shape}") # Must be [2, 206]

model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

Model loaded with 4,270,858 parameters
Dummy input shape: torch.Size([2, 1, 128, 313])
Dummy output shape: torch.Size([2, 206])


# Loss, Optimizer, Scheduler

In [22]:
import torch.nn as nn
import torch.optim as optim

# 1. Loss Function 
# We use CrossEntropyLoss which expects raw logits and integer targets
criterion = nn.CrossEntropyLoss()

# 2. Optimizer (AdamW is the standard for transformers/CNNs)
# lr=1e-3 is a good starting point for transfer learning
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

# 3. Scheduler (Cosine Annealing)
# T_max=5 means the learning rate will follow a cosine curve over 5 epochs
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)

# 4. Mixed Precision Scaler
# This cuts VRAM usage in half and speeds up training on T4 GPUs
# Note: In newer PyTorch versions, it's torch.amp.GradScaler('cuda')
try:
    scaler = torch.amp.GradScaler('cuda')
except AttributeError:
    scaler = torch.cuda.amp.GradScaler() # Fallback for older PyTorch

print("Criterion, Optimizer, Scheduler, and Scaler are ready!")

Criterion, Optimizer, Scheduler, and Scaler are ready!


# The Training Loop

In [23]:
import time

epochs = 5

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    start_time = time.time()
    
    # Loop over the batches
    for batch_idx, (mels, labs) in enumerate(train_loader):
        
        # 1. Move data to GPU
        mels = mels.to(device)
        labs = labs.to(device)
        mels = mels.unsqueeze(1)
        
        # 2. Zero gradients
        optimizer.zero_grad()
        
        # 3. Forward pass with Mixed Precision (Autocast)
        with torch.cuda.amp.autocast():
            outputs = model(mels)
            loss = criterion(outputs, labs)
            
        # 4. Backward pass with Scaler
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        # 5. Track loss
        running_loss += loss.item()
        
        # Print every 10 batches
        if (batch_idx + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{epochs}] | Batch [{batch_idx+1}/{len(train_loader)}] | Loss: {loss.item():.4f}")
            
    # --- END OF EPOCH ---
    # Step the scheduler (Crucial: only once per epoch!)
    scheduler.step()
    
    # Calculate average loss for the epoch
    avg_train_loss = running_loss / len(train_loader)
    end_time = time.time()
    
    print(f"=== Epoch {epoch+1} Complete | Avg Train Loss: {avg_train_loss:.4f} | Time: {end_time - start_time:.1f}s ===")
    print("-" * 50)

/tmp/ipykernel_22/2049029608.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch [1/5] | Batch [10/71] | Loss: 4.8420
Epoch [1/5] | Batch [20/71] | Loss: 4.6388
Epoch [1/5] | Batch [30/71] | Loss: 4.7538
Epoch [1/5] | Batch [40/71] | Loss: 4.6124
Epoch [1/5] | Batch [50/71] | Loss: 4.2225
Epoch [1/5] | Batch [60/71] | Loss: 4.1575
Epoch [1/5] | Batch [70/71] | Loss: 4.5109
=== Epoch 1 Complete | Avg Train Loss: 4.6404 | Time: 70.6s ===
--------------------------------------------------
Epoch [2/5] | Batch [10/71] | Loss: 3.0347
Epoch [2/5] | Batch [20/71] | Loss: 3.3019
Epoch [2/5] | Batch [30/71] | Loss: 3.2563
Epoch [2/5] | Batch [40/71] | Loss: 3.6311
Epoch [2/5] | Batch [50/71] | Loss: 3.5078
Epoch [2/5] | Batch [60/71] | Loss: 3.6114
Epoch [2/5] | Batch [70/71] | Loss: 3.0685
=== Epoch 2 Complete | Avg Train Loss: 3.3770 | Time: 40.6s ===
--------------------------------------------------
Epoch [3/5] | Batch [10/71] | Loss: 2.6844
Epoch [3/5] | Batch [20/71] | Loss: 2.0046
Epoch [3/5] | Batch [30/71] | Loss: 2.3131
Epoch [3/5] | Batch [40/71] | Loss: 2.1